# Silver Layer — Data Cleaning & Standardisation

## Retail Banking Environment — Customer Financial Behaviour Analysis

### Context
This notebook is **Part 2 of 3** in our group project medallion pipeline.

- **Part 1 (Bronze)** — Raw mock data was generated and written to `sandbox_catalog.banking_details` as four unprocessed tables: `customers`, `accounts`, `transactions`, `loan_applications`.
- **Part 2 (Silver) ← You are here** — Each Bronze table is read, inspected for quality issues, cleaned, and written back as a Silver table ready for joining.
- **Part 3 (Gold)** — Silver tables will be joined and aggregated into analytical Gold tables for EDA.

### What This Notebook Does
For each of the four tables we will:
1. Read the Bronze table and run a null report
2. Identify and document every data-quality issue injected at the Bronze stage
3. Apply targeted cleaning steps (drop nulls, standardise casing, fix inconsistent values, cast types, filter invalid rows)
4. Write the cleaned result as a Silver Delta table

> **Note for the Gold layer team member:** All Silver tables share the same catalog and schema (`sandbox_catalog.banking_details`). Table names follow the pattern `<entity>_silver` (e.g. `customers_silver`). Foreign key columns (`customer_id`, `account_id`) are preserved as-is so joins work directly.

---
## Imports & Configuration

In [0]:
import logging
import re

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType

# ── Logging setup ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("silver")

# ── Catalog / schema constants ─────────────────────────────────────────────────
# Must match the Bronze notebook exactly
CATALOG = "sandbox_catalog"
SCHEMA  = "banking_details"

log.info("Imports complete. Catalog=%s  Schema=%s", CATALOG, SCHEMA)

---
## Helper Functions

Reusable utilities used throughout this notebook. Define them once here and call them for every table below.

In [0]:
def read_bronze(table_name: str):
    """
    Read a Bronze Delta table from the active catalog/schema.
    Returns a Spark DataFrame and logs the row count.
    """
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df = spark.read.table(full_name)
    log.info("Loaded Bronze <- %s  (%d rows)", full_name, df.count())
    return df


def write_silver(df, table_name: str, mode: str = "overwrite") -> None:
    """
    Write a cleaned Spark DataFrame as a Silver Delta table.
    Uses overwrite mode by default so the notebook is safely re-runnable.
    """
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df.write.format("delta").mode(mode).saveAsTable(full_name)
    log.info("Written Silver -> %s  (%d rows)", full_name, df.count())


def null_report(df, label: str) -> None:
    """
    Print a per-column null count for the given DataFrame.
    Useful for inspecting raw Bronze tables before cleaning.
    """
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])
    print(f"\n── Null Report: {label} ──")
    null_counts.show()


def clean_column_names(df):
    """
    Lowercase all column names and replace spaces / special characters
    with underscores. Ensures consistent naming across all Silver tables.
    """
    new_cols = [
        re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_").lower()
        for c in df.columns
    ]
    return df.toDF(*new_cols)


log.info("Helper functions defined.")

---
## 1. Customers — Silver Cleaning

### Known Bronze Data-Quality Issues
| Column | Issue |
|---|---|
| `province` | Mixed formats — full name, abbreviation, and lowercase variants all exist (e.g. `"Alberta"`, `"AB"`, `"alberta"`) |
| `age` | No explicit nulls injected, but ages outside 18–100 should be treated as invalid |
| General | Column names are already clean; no structural issues |

### Cleaning Steps
1. Run null report on the raw Bronze table
2. Drop any rows where a required key column (`customer_id`) is null
3. Standardise `province` to the two-letter uppercase abbreviation
4. Filter out any ages outside the plausible range (18–100)
5. Write to `customers_silver`

In [0]:
# ── 1.1  Load & Inspect ───────────────────────────────────────────────────────
df_customers_bronze = read_bronze("customers")
null_report(df_customers_bronze, "customers (Bronze)")
display(df_customers_bronze)

In [0]:
# ── 1.2  Standardise Province ─────────────────────────────────────────────────
# The Bronze table contains all three formats for every province:
#   full name (e.g. "Alberta"), abbreviation ("AB"), lowercase ("alberta")
# We map every known variant to the official two-letter uppercase code.

province_map = {
    # Alberta
    "alberta": "AB", "ab": "AB", "Alberta": "AB", "AB": "AB",
    # British Columbia
    "british columbia": "BC", "bc": "BC", "British Columbia": "BC", "BC": "BC",
    # Manitoba
    "manitoba": "MB", "mb": "MB", "Manitoba": "MB", "MB": "MB",
    # New Brunswick
    "new brunswick": "NB", "nb": "NB", "New Brunswick": "NB", "NB": "NB",
    # Newfoundland and Labrador
    "newfoundland and labrador": "NL", "nl": "NL",
    "Newfoundland and Labrador": "NL", "NL": "NL",
    # Nova Scotia
    "nova scotia": "NS", "ns": "NS", "Nova Scotia": "NS", "NS": "NS",
    # Ontario
    "ontario": "ON", "on": "ON", "Ontario": "ON", "ON": "ON",
    # Prince Edward Island
    "prince edward island": "PE", "pe": "PE",
    "Prince Edward Island": "PE", "PE": "PE",
    # Quebec
    "quebec": "QC", "qc": "QC", "Quebec": "QC", "QC": "QC",
    # Saskatchewan
    "saskatchewan": "SK", "sk": "SK", "Saskatchewan": "SK", "SK": "SK",
}

# Build a Spark mapping expression from the dictionary above
province_mapping_expr = F.create_map(
    *[item for pair in province_map.items() for item in (F.lit(pair[0]), F.lit(pair[1]))]
)

df_customers_clean = (
    df_customers_bronze
    # Drop rows with a null customer_id (primary key must not be null)
    .filter(F.col("customer_id").isNotNull())

    # Standardise province using the mapping; unrecognised values become null
    .withColumn("province", province_mapping_expr[F.col("province")])

    # Drop rows where province could not be mapped (unexpected / truly bad values)
    .filter(F.col("province").isNotNull())

    # Filter out implausible ages (must be 18–100 inclusive)
    .filter(F.col("age").between(18, 100))

    # Ensure income_bracket is not null (required categorical field)
    .filter(F.col("income_bracket").isNotNull())

    # Ensure join_date is not null
    .filter(F.col("join_date").isNotNull())
)

# ── 1.3  Verify & Write ───────────────────────────────────────────────────────
null_report(df_customers_clean, "customers (Silver — post-cleaning)")
display(df_customers_clean)

write_silver(df_customers_clean, "customers_silver")

---
## 2. Accounts — Silver Cleaning

### Known Bronze Data-Quality Issues
| Column | Issue |
|---|---|
| `account_type` | Inconsistent casing and spelling — `'savings'`, `'Savings'`, `'CHECKINGS'`, `'Checkings'` |
| `status` | All lowercase already (`'active'`, `'inactive'`, `'closed'`), but should be validated against the known set |
| `balance` | Randomly generated between 1,000–1,000,000; no explicit invalid values, but zero/negative balances should be filtered |
| General | No nulls explicitly injected, but structural completeness check is still performed |

### Cleaning Steps
1. Run null report
2. Drop rows with null `account_id` or `customer_id`
3. Standardise `account_type` to lowercase (`'chequing'` / `'savings'`)
4. Validate `status` against the known set
5. Filter out non-positive balances
6. Write to `accounts_silver`

In [0]:
# ── 2.1  Load & Inspect ───────────────────────────────────────────────────────
df_accounts_bronze = read_bronze("accounts")
null_report(df_accounts_bronze, "accounts (Bronze)")
display(df_accounts_bronze)

In [0]:
# ── 2.2  Clean ────────────────────────────────────────────────────────────────

# Valid status values for an account
VALID_STATUSES = ["active", "inactive", "closed"]

df_accounts_clean = (
    df_accounts_bronze
    # Drop rows missing either primary or foreign key
    .filter(F.col("account_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())

    # Standardise account_type:
    #   'CHECKINGS' / 'Checkings'  → 'chequing'  (Canadian spelling)
    #   'Savings'   / 'savings'    → 'savings'
    .withColumn(
        "account_type",
        F.when(
            F.lower(F.col("account_type")).isin("checkings", "chequing"), F.lit("chequing")
        ).when(
            F.lower(F.col("account_type")) == "savings", F.lit("savings")
        ).otherwise(None)   # any unrecognised value becomes null and will be dropped
    )

    # Drop rows with an unrecognised account_type
    .filter(F.col("account_type").isNotNull())

    # Standardise status to lowercase and validate against the known set
    .withColumn("status", F.lower(F.col("status")))
    .filter(F.col("status").isin(VALID_STATUSES))

    # Remove rows with non-positive balances (business rule: balance must be > 0)
    .filter(F.col("balance") > 0)

    # Ensure open_date is present
    .filter(F.col("open_date").isNotNull())
)

# ── 2.3  Verify & Write ───────────────────────────────────────────────────────
null_report(df_accounts_clean, "accounts (Silver — post-cleaning)")
display(df_accounts_clean)

write_silver(df_accounts_clean, "accounts_silver")

---
## 3. Transactions — Silver Cleaning

### Known Bronze Data-Quality Issues
| Column | Issue |
|---|---|
| `amount` | Stored as a **string** instead of a numeric type; ~3% of rows are `null`; ~5% of withdrawals have a **negative string value** (e.g. `"-123.45"`) |
| `merchant_category` | Intentionally `null` for `deposit` and `transfer` types; also randomly null for ~5% of withdrawals — these nulls for non-withdrawal types are expected, not errors |
| `transaction_date` | Stored as a string (`"YYYY-MM-DD"`) — must be cast to `DateType` |
| General | Rows with null `amount` should be dropped; negative amounts on withdrawals should be converted to their absolute value |

### Cleaning Steps
1. Run null report
2. Drop rows with null `transaction_id` or `account_id`
3. Drop rows where `amount` is null (genuinely missing data)
4. Cast `amount` from string to `DoubleType`
5. Convert any negative amounts to their absolute value (the sign is captured by `transaction_type`)
6. Cast `transaction_date` from string to `DateType`
7. Validate `transaction_type` against the known set
8. Write to `transactions_silver`

In [0]:
# ── 3.1  Load & Inspect ───────────────────────────────────────────────────────
df_transactions_bronze = read_bronze("transactions")
null_report(df_transactions_bronze, "transactions (Bronze)")
display(df_transactions_bronze)

In [0]:
# ── 3.2  Clean ────────────────────────────────────────────────────────────────

VALID_TRANSACTION_TYPES = ["deposit", "withdrawal", "transfer"]

df_transactions_clean = (
    df_transactions_bronze
    # Drop rows missing the primary or foreign key
    .filter(F.col("transaction_id").isNotNull())
    .filter(F.col("account_id").isNotNull())

    # Drop rows where amount is null — these represent genuinely missing data
    # (~3% of rows per the Bronze generation logic)
    .filter(F.col("amount").isNotNull())

    # Cast amount from string to double.
    # This will produce null for any non-numeric strings; those are then dropped.
    .withColumn("amount", F.col("amount").cast(DoubleType()))
    .filter(F.col("amount").isNotNull())

    # Negative amounts on withdrawals are a formatting artefact from Bronze.
    # We take the absolute value — the direction of money flow is
    # already captured by transaction_type.
    .withColumn("amount", F.abs(F.col("amount")))

    # Filter out zero amounts (no valid transaction has a zero value)
    .filter(F.col("amount") > 0)

    # Cast transaction_date from string ("YYYY-MM-DD") to proper DateType
    .withColumn("transaction_date", F.to_date(F.col("transaction_date"), "yyyy-MM-dd"))
    .filter(F.col("transaction_date").isNotNull())   # drop rows with unparseable dates

    # Validate transaction_type against the known set
    .filter(F.col("transaction_type").isin(VALID_TRANSACTION_TYPES))

    # NOTE: null merchant_category is expected for deposits and transfers.
    # We do NOT drop these rows — the null is semantically correct.
)

In [0]:
# ── 3.3  Verify & Write ───────────────────────────────────────────────────────
# merchant_category will still show nulls in the report — this is expected.
null_report(df_transactions_clean, "transactions (Silver — post-cleaning)")
display(df_transactions_clean)

write_silver(df_transactions_clean, "transactions_silver")

---
## 4. Loan Applications — Silver Cleaning

### Known Bronze Data-Quality Issues
| Column | Issue |
|---|---|
| `amount_requested` | ~5% of rows are `null` — must be dropped |
| `interest_rate` | ~3% of rows have clearly invalid rates between 100% and 200% — must be filtered out |
| `application_date` | Stored as a string (`"YYYY-MM-DD"`) — must be cast to `DateType` |
| `loan_type` | All lowercase already; validate against the known set (`mortgage`, `auto`, `personal`) |
| `status` | All lowercase already; validate against the known set (`approved`, `denied`, `pending`) |

### Cleaning Steps
1. Run null report
2. Drop rows with null `application_id` or `customer_id`
3. Drop rows where `amount_requested` is null
4. Filter out invalid interest rates (> 30% is treated as implausible for any loan type)
5. Cast `application_date` from string to `DateType`
6. Validate `loan_type` and `status` against known sets
7. Write to `loan_applications_silver`

In [0]:
# ── 4.1  Load & Inspect ───────────────────────────────────────────────────────
df_loans_bronze = read_bronze("loan_applications")
null_report(df_loans_bronze, "loan_applications (Bronze)")
display(df_loans_bronze)

In [0]:
# ── 4.2  Clean ────────────────────────────────────────────────────────────────

VALID_LOAN_TYPES   = ["mortgage", "auto", "personal"]
VALID_LOAN_STATUSES = ["approved", "denied", "pending"]

# Maximum plausible interest rate (%).
# Bronze injected values between 100–200% as clearly invalid records.
MAX_INTEREST_RATE = 30.0

df_loans_clean = (
    df_loans_bronze
    # Drop rows missing the primary or foreign key
    .filter(F.col("application_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())

    # Drop rows where the requested amount is missing (~5% of Bronze rows)
    .filter(F.col("amount_requested").isNotNull())

    # Filter out negative or zero loan amounts (not financially meaningful)
    .filter(F.col("amount_requested") > 0)

    # Remove rows with clearly invalid interest rates (> 30%)
    # The Bronze layer intentionally injected rates between 100–200% as bad data
    .filter(F.col("interest_rate") <= MAX_INTEREST_RATE)

    # Also drop rows where interest_rate is null or negative
    .filter(F.col("interest_rate").isNotNull())
    .filter(F.col("interest_rate") > 0)

    # Cast application_date from string ("YYYY-MM-DD") to proper DateType
    .withColumn("application_date", F.to_date(F.col("application_date"), "yyyy-MM-dd"))
    .filter(F.col("application_date").isNotNull())  # drop rows with unparseable dates

    # Validate loan_type and status against the known sets
    .filter(F.col("loan_type").isin(VALID_LOAN_TYPES))
    .filter(F.col("status").isin(VALID_LOAN_STATUSES))
)

# ── 4.3  Verify & Write ───────────────────────────────────────────────────────
null_report(df_loans_clean, "loan_applications (Silver — post-cleaning)")
display(df_loans_clean)

write_silver(df_loans_clean, "loan_applications_silver")

---
## 5. Silver Layer Summary

Quick sanity-check: reload all four Silver tables and print their row counts and schemas side-by-side. This confirms the writes succeeded and gives the Gold layer team member a clear picture of what is available.

In [0]:
# ── 5.1  Reload all Silver tables ─────────────────────────────────────────────
silver_tables = [
    "customers_silver",
    "accounts_silver",
    "transactions_silver",
    "loan_applications_silver",
]

for table in silver_tables:
    full_name = f"{CATALOG}.{SCHEMA}.{table}"
    df = spark.read.table(full_name)
    print(f"\n{'='*60}")
    print(f"  TABLE : {full_name}")
    print(f"  ROWS  : {df.count():,}")
    print(f"{'='*60}")
    df.printSchema()

In [0]:
# ── 5.2  Quick display of each Silver table ───────────────────────────────────
# Run a final null report on all Silver tables to confirm cleanliness.
# merchant_category in transactions_silver will still show nulls — this is expected.

df_c = spark.read.table(f"{CATALOG}.{SCHEMA}.customers_silver")
df_a = spark.read.table(f"{CATALOG}.{SCHEMA}.accounts_silver")
df_t = spark.read.table(f"{CATALOG}.{SCHEMA}.transactions_silver")
df_l = spark.read.table(f"{CATALOG}.{SCHEMA}.loan_applications_silver")

null_report(df_c, "customers_silver")
null_report(df_a, "accounts_silver")
null_report(df_t, "transactions_silver")
null_report(df_l, "loan_applications_silver")

log.info("Silver layer complete. All four tables are ready for the Gold layer.")

---
## Handoff Notes for the Gold Layer

All Silver tables live in `sandbox_catalog.banking_details`. Here is a quick reference for building joins:

| Silver Table | Primary Key | Foreign Key | Notes |
|---|---|---|---|
| `customers_silver` | `customer_id` | — | Province is now a 2-letter code (e.g. `"ON"`) |
| `accounts_silver` | `account_id` | `customer_id` | `account_type` is `'chequing'` or `'savings'` |
| `transactions_silver` | `transaction_id` | `account_id` | `amount` is positive `DoubleType`; `transaction_date` is `DateType`; null `merchant_category` is expected for deposits/transfers |
| `loan_applications_silver` | `application_id` | `customer_id` | `application_date` is `DateType`; interest rates are capped at 30% |